# Brazil's Marketplace in Motion
### A data story of the Olist e-commerce ecosystem (2016–2018)

**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — 100k real orders placed across Olist's multi-seller marketplace, spanning order status, price, payments, freight, customer & seller geolocation, products, and customer reviews. Sourced directly from Olist's own GitHub repository (`olist/work-at-olist-data`), an official mirror of the Kaggle release.

**Why this dataset:** it is real commercial data (anonymised), genuinely rich (9 linked tables, 45+ usable fields after merging), and varied — numerical (price, freight, weight), categorical (product category, payment type, state), spatial (customer/seller state & zip), and temporal (order timestamps 2016–2018).

**Structure of this notebook:**
1. Data loading & preliminary EDA
2. 12 analytical questions, each answered with one publication-ready Plotly visualization
3. Summary of key takeaways


In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import sys, os
sys.path.insert(0, os.path.dirname((os.path.abspath(''))))
from utils import *

pd.set_option("display.max_columns", 50)
pio.defaults.default_format = "png"   # static, high-res images so the notebook renders identically in Jupyter, HTML and PDF exports
pio.defaults.default_scale = 2   

# ---- CVD-safe palette (Okabe-Ito) -----------------------------------
GREY      = "#B0B0B0"   # muted context
HIGHLIGHT = "#0072B2"   # focus blue
ACCENT2   = "#D55E00"   # vermillion (secondary highlight)
ACCENT3   = "#009E73"   # bluish green
ACCENT4   = "#E69F00"   # orange
CVD_SEQ   = ["#0072B2", "#56B4E9", "#009E73", "#E69F00", "#D55E00", "#CC79A7", "#F0E442"]

TEMPLATE = "plotly_white"

def style(fig, title, height=460, width=900):
    fig.update_layout(
        template=TEMPLATE,
        title=dict(text=title, font=dict(size=17, family="Arial", color="#222"), x=0.02, xanchor="left"),
        font=dict(family="Arial", size=13, color="#333"),
        height=height, width=width,
        margin=dict(t=70, l=60, r=40, b=60),
        plot_bgcolor="white", paper_bgcolor="white",
        legend=dict(bgcolor="rgba(0,0,0,0)"),
    )
    fig.update_xaxes(showgrid=False, zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor="#EDEDED", zeroline=False)
    return fig


## 1 · Load data
Using the pre-merged master table (see `prepare_data.py` in this folder for the full merge/clean pipeline across the 9 raw Olist tables).

In [10]:
items = pd.read_parquet("../data/master_items.parquet")
delivered = pd.read_parquet("../data/delivered_items.parquet")   # delivered orders only (needed for delivery-time analysis)
customers_summary = pd.read_parquet("../data/customers_summary.parquet")

print(f"{items.shape[0]:,} order-items | {items['order_id'].nunique():,} orders | "
      f"{items['customer_unique_id'].nunique():,} unique customers | "
      f"{items['seller_id'].nunique():,} sellers | {items['product_category_name_english'].nunique()} product categories")
items.head(3)

112,650 order-items | 98,666 orders | 95,420 unique customers | 3,095 sellers | 72 product categories


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category_name_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,total_payment_value,n_payment_types,max_installments,main_payment_type,review_score,review_creation_date,customer_state_name,seller_state_name,item_total,freight_ratio,purchase_month,purchase_year,purchase_dow,purchase_hour,delivery_days,delivery_delay_days,is_late,cross_state_shipment,product_volume_cm3
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,cool_stuff,650.0,28.0,9.0,14.0,27277,volta redonda,SP,72.19,1.0,2.0,credit_card,5.0,2017-09-21,Rio de Janeiro,São Paulo,72.19,0.225637,2017-09,2017,Wednesday,8,7.0,-9.0,False,True,3528.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,pet_shop,30000.0,50.0,30.0,40.0,3471,sao paulo,SP,259.83,1.0,3.0,credit_card,4.0,2017-05-13,São Paulo,São Paulo,259.83,0.083076,2017-04,2017,Wednesday,10,16.0,-3.0,False,False,60000.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,furniture_decor,3050.0,33.0,13.0,33.0,37564,borda da mata,MG,216.87,1.0,5.0,credit_card,5.0,2018-01-23,Minas Gerais,Minas Gerais,216.87,0.089799,2018-01,2018,Sunday,14,7.0,-14.0,False,False,14157.0


## 2 · Preliminary Exploratory Data Analysis

A quick look at shape, missingness, and single-variable summaries before moving to the analytical questions (these are exploratory only — not counted among the 12 questions below).

In [11]:
items.dtypes.value_counts()

str               16
float64           15
datetime64[us]     7
int64              3
int32              2
bool               2
Name: count, dtype: int64

In [12]:
items.isna().mean().sort_values(ascending=False).head(10).round(3)

order_delivered_customer_date    0.022
delivery_delay_days              0.022
delivery_days                    0.022
order_delivered_carrier_date     0.011
review_creation_date             0.008
review_score                     0.008
product_width_cm                 0.000
product_length_cm                0.000
product_weight_g                 0.000
product_height_cm                0.000
dtype: float64

In [13]:
items[['price','freight_value','item_total','review_score']].describe().round(2)

,price,freight_value,item_total,review_score
count,112650.00,112650.00,112650.00,111708.00
mean,120.65,20.03,140.69,4.03
std,183.63,16.02,190.76,1.39
min,0.85,0.00,6.08,1.00
25%,39.90,13.08,55.23,4.00
50%,74.99,16.27,92.36,5.00
75%,134.90,21.15,157.95,5.00
max,6735.00,409.68,6929.31,5.00


In [14]:
items['order_status'].value_counts()

order_status
delivered      110197
shipped          1185
canceled          542
invoiced          359
processing        357
unavailable         7
approved            3
Name: count, dtype: int64

In [15]:
print("Date range:", items['order_purchase_timestamp'].min(), "to", items['order_purchase_timestamp'].max())

Date range: 2016-09-04 21:15:19 to 2018-09-03 09:06:57


## 3 · Analytical Questions

Each question below is multi-dimensional — relating variables, comparing groups, or tracking change over time/space — and answered with one dedicated, explanatory Plotly visualization.

### Q1 — How does average delivery time vary across Brazilian states, and does slower delivery come with lower customer satisfaction?

In [16]:
state_perf = (delivered.groupby('customer_state', as_index=False)
              .agg(avg_delivery_days=('delivery_days','mean'),
                   avg_review=('review_score','mean'),
                   n_orders=('order_id','nunique'))
              .query('n_orders >= 30')
              .sort_values('avg_delivery_days'))

fig = px.scatter(state_perf, x='avg_delivery_days', y='avg_review', size='n_orders',
                  color='avg_delivery_days', color_continuous_scale=[BLUE, RED],
                  hover_name='customer_state', size_max=45, 
                  labels={'avg_delivery_days': 'Average delivery time (days)', 'avg_review': 'Average review score (1-5)'})
fig.update_traces(marker=dict(line=dict(width=1, color='black')))
fig.add_hline(y=state_perf['avg_review'].mean(), line_dash='dot', line_color=BLACK,
              annotation_text='national average', annotation_position='top left')
fig = style(fig, 'Slower-delivery states also rate their orders lower', height=480)
fig.update_coloraxes(showscale=False)
fig.show()

### Q2 — Which product categories drive the most revenue, and how has monthly revenue trended over 2017–2018?

In [17]:
top_cats = (items.groupby('product_category_name_english')['item_total'].sum()
            .sort_values(ascending=False).head(8).index)

monthly_cat = (items[items['product_category_name_english'].isin(top_cats) &
                      items['purchase_month'].between('2017-01','2018-08')]
               .groupby(['purchase_month','product_category_name_english'])['item_total']
               .sum().reset_index())

fig = px.line(monthly_cat, x='purchase_month', y='item_total', color='product_category_name_english',
              color_discrete_sequence=CVD_SEQ,
              labels={'purchase_month': 'Month', 'item_total': 'Revenue (R$)', 'product_category_name_english': 'Category'})
fig.update_traces(line=dict(width=2.5))
fig = style(fig, 'Revenue trend for the top categories', height=480)
fig.update_xaxes(tickangle=-45)
fig.show()

### Q3 — How does the freight-to-price ratio differ across product categories, and which categories are costliest to ship relative to their value?

In [18]:
cat_freight = (delivered.groupby('product_category_name_english')
               .agg(median_ratio=('freight_ratio','median'), n=('order_id','count'))
               .query('n >= 100').sort_values('median_ratio', ascending=False).head(12).reset_index())

fig = px.bar(cat_freight.sort_values('median_ratio'), x='median_ratio', y='product_category_name_english',
             orientation='h', color='median_ratio', color_continuous_scale=[GREY, BLUE],
             labels={'median_ratio': 'Median freight ÷ price', 'product_category_name_english': ''})
fig = style(fig, 'Bulky, low-value categories carry the heaviest relative freight cost', height=500)
fig.update_coloraxes(showscale=False)
fig.update_xaxes(tickformat='.0%')
fig.update_layout(margin=dict(l=200))
fig.show()

### Q4 — Do orders that arrive later than the estimated delivery date get lower review scores?

In [19]:
d = delivered.dropna(subset=['delivery_delay_days','review_score']).copy()
d['delay_bucket'] = pd.cut(d['delivery_delay_days'],
                            bins=[-100,-7,-1,0,7,100],
                            labels=['7+ days early','1-6 days early','on time','1-7 days late','7+ days late'])
bucket_rev = d.groupby('delay_bucket', observed=True)['review_score'].mean().reset_index()

colors = [GREEN, GREEN, GREEN, ORANGE, DARKRED]
fig = px.bar(bucket_rev, x='delay_bucket', y='review_score',
             labels={'delay_bucket': 'Delivery timing vs. estimate', 'review_score': 'Average review score'})
fig.update_traces(marker_color=colors[:len(bucket_rev)],
                   text=bucket_rev['review_score'].round(2), textposition='outside')
fig.add_hline(y=d['review_score'].mean(), line_dash='dot', line_color='#999')
fig = style(fig, 'A late delivery costs almost a full star', height=460)
fig.update_yaxes(range=[0, 5.3])
fig.show()

### Q5 — Which states have the highest average order value, and does typical installment count vary by region?

In [20]:
state_value = (items.groupby('customer_state', as_index=False)
               .agg(avg_order_value=('item_total','mean'),
                    avg_installments=('max_installments','mean'),
                    n=('order_id','nunique'))
               .query('n >= 30').sort_values('avg_order_value', ascending=False).head(15))

fig = go.Figure()
fig.add_bar(x=state_value['customer_state'], y=state_value['avg_order_value'], name='Avg order value (R$)',
            marker_color=BLUE, yaxis='y')
fig.add_scatter(x=state_value['customer_state'], y=state_value['avg_installments'], name='Avg installments',
                 mode='lines+markers', marker=dict(color=ORANGE, size=8), line=dict(width=2), yaxis='y2')
fig.update_layout(
    yaxis=dict(title='Avg order value (R$)'),
    yaxis2=dict(title='Avg installments chosen', overlaying='y', side='right', showgrid=False),
    legend=dict(orientation='h', y=20, x=0.02))
fig.show()

### Q6 — How do review scores relate to price tier, and does that relationship hold across product categories?

In [21]:
d = delivered.dropna(subset=['review_score']).copy()
d['price_tier'] = pd.qcut(d['price'], 4, labels=['Q1 (cheapest)','Q2','Q3','Q4 (priciest)'])
top6 = items.groupby('product_category_name_english')['item_total'].sum().sort_values(ascending=False).head(6).index
d6 = d[d['product_category_name_english'].isin(top6)]

pivot = d6.groupby(['product_category_name_english','price_tier'], observed=True)['review_score'].mean().reset_index()

fig = px.bar(pivot, x='product_category_name_english', y='review_score', color='price_tier', barmode='group',
             color_discrete_sequence=CVD_SEQ,
             labels={'product_category_name_english': '', 'review_score': 'Average review score', 'price_tier': 'Price tier'})
fig = style(fig, "Higher price tiers don't buy higher satisfaction", height=480)
fig.update_yaxes(range=[max(0, pivot['review_score'].min()-0.3), min(5, pivot['review_score'].max())])
fig.update_xaxes(tickangle=-20)
fig.show()

### Q7 — What's the weekly rhythm of order placement, and has it shifted between 2017 and 2018?

In [22]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekly = (items[items['purchase_year'].isin([2017,2018])]
          .groupby(['purchase_year','purchase_dow'])['order_id'].nunique().reset_index())
weekly['purchase_dow'] = pd.Categorical(weekly['purchase_dow'], categories=dow_order, ordered=True)
weekly = weekly.sort_values('purchase_dow')

fig = px.line(weekly, x='purchase_dow', y='order_id', color='purchase_year',
              color_discrete_sequence=[GREY, HIGHLIGHT], markers=True,
              labels={'purchase_dow': 'Day of week', 'order_id': 'Number of orders', 'purchase_year': 'Year'})
fig.update_traces(line=dict(width=2.5), marker=dict(size=8))
fig = style(fig, 'Weekday order volume grows faster than weekends year over year', height=460)
fig.show()

### Q8 — Does paying in more installments correlate with higher order value, and does that differ by payment type?

In [23]:
d = items.dropna(subset=['main_payment_type']).query("main_payment_type != 'not_defined'")
inst = (d.groupby(['main_payment_type','max_installments'], as_index=False)
        .agg(avg_value=('item_total','mean'), n=('order_id','nunique'))
        .query('n >= 20 and max_installments <= 12'))

fig = px.scatter(inst, x='max_installments', y='avg_value', color='main_payment_type', size='n',
                  color_discrete_sequence=CVD_SEQ, size_max=35,
                  labels={'max_installments': 'Number of installments', 'avg_value': 'Average order value (R$)',
                          'main_payment_type': 'Payment type'})
fig = style(fig, 'More installments track with bigger baskets — most visibly on credit cards', height=480)
fig.show()

### Q9 — How concentrated is revenue across sellers — do a small number of sellers dominate the marketplace?

In [25]:
seller_rev = items.groupby('seller_id')['item_total'].sum().sort_values(ascending=False).reset_index()
seller_rev['cum_share'] = seller_rev['item_total'].cumsum() / seller_rev['item_total'].sum()
seller_rev['seller_rank_pct'] = (np.arange(1, len(seller_rev)+1) / len(seller_rev)) * 100
top20_share = seller_rev.loc[seller_rev['seller_rank_pct'] <= 20, 'item_total'].sum() / seller_rev['item_total'].sum()

fig = go.Figure()
fig.add_scatter(x=seller_rev['seller_rank_pct'], y=seller_rev['cum_share'] * 100, mode='lines',
                 line=dict(color=BLUE, width=3), fill='tozeroy', fillcolor='rgba(46,117,182,0.12)')
fig.add_annotation(x=20, y=top20_share * 100, text=f"top 20% of sellers = {top20_share:.0%} of revenue",
                    showarrow=True, arrowhead=2, ax=60, ay=-40, font=dict(color=ORANGE))
fig = style(fig, 'A small slice of sellers accounts for most marketplace revenue', height=480)
fig.update_xaxes(title='Sellers ranked by revenue (cumulative %)')
fig.update_yaxes(title='Cumulative share of revenue (%)')
fig.update_layout(showlegend=False)
fig.show()

### Q10 — How often are orders shipped cross-state, and does that vary by customer region?

In [26]:
cross = (items.groupby('customer_state', as_index=False)
         .agg(pct_cross_state=('cross_state_shipment','mean'), n=('order_id','nunique'))
         .query('n >= 30').sort_values('pct_cross_state', ascending=False))
cross['pct_cross_state'] *= 100

fig = px.bar(cross, x='customer_state', y='pct_cross_state', color='pct_cross_state',
             color_continuous_scale=[GREEN, GREY],
             labels={'customer_state': 'Customer state', 'pct_cross_state': '% of orders shipped from another state'})
fig.add_hline(y=cross['pct_cross_state'].mean(), line_dash='dot', line_color='#666',
              annotation_text='national average', annotation_position='top right')
fig = style(fig, 'Outside the São Paulo–Rio–Minas core, most orders ship in from elsewhere', height=480)
fig.update_coloraxes(showscale=False)
fig.show()

### Q11 — How does product size (volume) relate to freight cost, and does this relationship differ for the largest product categories?

In [27]:
d = delivered.dropna(subset=['product_volume_cm3','freight_value']).copy()
d = d[(d['product_volume_cm3']>0) & (d['product_volume_cm3'] < d['product_volume_cm3'].quantile(0.98))]
top5 = items.groupby('product_category_name_english')['item_total'].sum().sort_values(ascending=False).head(5).index
d5 = d[d['product_category_name_english'].isin(top5)]

fig = px.scatter(d5, x='product_volume_cm3', y='freight_value', color='product_category_name_english',
                  trendline='ols', opacity=0.35, color_discrete_sequence=CVD_SEQ,
                  labels={'product_volume_cm3': 'Product volume (cm³)', 'freight_value': 'Freight cost (R$)',
                          'product_category_name_english': 'Category'})
fig = style(fig, 'Freight scales with product volume — steepest for furniture & bed/bath/table', height=480)
fig.show()

### Q12 — What share of customers are repeat buyers, and do repeat customers spend more or rate their experience differently?

In [28]:
seg = (customers_summary.assign(segment=lambda x: np.where(x['is_repeat'],'Repeat customer','One-time customer'))
       .groupby('segment')
       .agg(n_customers=('customer_unique_id','count'),
            avg_spent=('total_spent','mean'),
            avg_review=('avg_review','mean'))
       .reset_index())
seg['pct_of_customers'] = seg['n_customers'] / seg['n_customers'].sum() * 100

fig = make_subplots(rows=1, cols=2, subplot_titles=('Share of customer base (%)', 'Avg. spend per customer (R$)'))
fig.add_trace(go.Bar(x=seg['segment'], y=seg['pct_of_customers'], marker_color=[GREY, BLUE],
                      text=seg['pct_of_customers'].round(1), textposition='outside'), row=1, col=1)
fig.add_trace(go.Bar(x=seg['segment'], y=seg['avg_spent'], marker_color=[GREY, BLUE],
                      text=seg['avg_spent'].round(0), textposition='outside'), row=1, col=2)
fig = style(fig, 'Repeat customers are rare but spend meaningfully more per head', height=440)
fig.update_layout(showlegend=False)
fig.show()

print(seg.round(2))

             segment  n_customers  avg_spent  avg_review  pct_of_customers
0  One-time customer        92507     161.53        4.10             96.95
1    Repeat customer         2913     310.75        4.14              3.05


## 4 · Key takeaways

1. **Delivery speed drives satisfaction more than anything else** — states with longer average delivery times post visibly lower review scores, and orders that arrive even one day late lose the better part of a star versus on-time orders.
2. **Revenue is geographically and structurally concentrated.** São Paulo dominates as both the commercial hub and main shipping origin — most other states rely on cross-state shipments — and the top ~20% of sellers account for the large majority of marketplace revenue.
3. **Price does not buy satisfaction.** Within the best-selling categories, review scores stay essentially flat across price tiers — Olist's satisfaction problem is operational (delivery, freight), not about product pricing.
4. **Freight cost is a size/weight game.** Bulky, lower-value categories like furniture and construction materials carry the worst freight-to-price ratio, and freight cost scales fastest with volume for exactly these categories.
5. **Growth in 2018 was healthy and increasingly weekday-driven**, led by health & beauty and watches/gifts, while installment financing — especially on credit cards — keeps pace with basket size, most strongly in the North.
6. **Repeat purchase is rare (~3% of customers) but valuable** — repeat customers spend more per head, making retention (closely tied to delivery reliability) a clear lever for revenue growth.

*(See the accompanying Streamlit dashboard for an interactive exploration of these findings, and the presentation deck for a slide-level summary.)*
